In [1]:
import os, re, json, textwrap
from pathlib import Path
import numpy as np
import requests
import chromadb

try:
    from tqdm import tqdm          # progress bar for the one-time embed (optional)
except ImportError:
    tqdm = None

# --- config (same values as the core notebook, so the store matches whether we
#     reuse the one it built or build an identical one ourselves) ---
OLLAMA_URL  = "http://localhost:11434"
CHAT_MODEL  = "llama3.1:8b"
EMBED_MODEL = "embeddinggemma:latest"
TOPK        = 5

WORDS_PER_CHUNK  = 300       # only used on the self-build path below
OVERLAP_WORDS    = 60
EMBED_BATCH_SIZE = 64
CORPUS_DIR        = Path("corpus_jupyter")
CHROMA_PATH       = "./chroma_db"
CHROMA_COLLECTION = "rag_demo"
DOWNLOAD_FROM_WEB = True

SESSION = requests.Session()
SESSION.trust_env = False        # bypass VPN/proxy for the localhost Ollama calls

# --- open the SAME collection the core notebook uses (the foundation we reuse) ---
collection = chromadb.PersistentClient(path=CHROMA_PATH).get_or_create_collection(
    CHROMA_COLLECTION, metadata={"hnsw:space": "cosine"})

already_built = collection.count() > 0
print(f"Chroma collection '{CHROMA_COLLECTION}': {collection.count()} vectors.")
print("Already populated — we'll reuse it." if already_built
      else "Empty — the next cell will build it from scratch.")

Chroma collection 'rag_demo': 8509 vectors.
Already populated — we'll reuse it.


In [2]:
if already_built:
    print(f"Reusing '{CHROMA_COLLECTION}' — {collection.count()} vectors already on disk. "
          "Nothing to build.")
else:
    print("Empty collection — building it now (same pipeline as the core notebook)...\n")

    GUTENBERG_BOOKS = {
        "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
        "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
        "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
        "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
        "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
        "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
        "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
        "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
        "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
        "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
        "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
        "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
        "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",
    }

    # 1. Download each book; strip Project Gutenberg's license header/footer.
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
    END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

    docs = []
    for title, url in GUTENBERG_BOOKS.items():
        path = CORPUS_DIR / (re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_") + ".txt")
        if not path.exists():
            if not DOWNLOAD_FROM_WEB:
                continue
            print(f"  downloading: {title}")
            resp = SESSION.get(url, timeout=180)
            resp.raise_for_status()
            raw = resp.text
            s, e = START_MARK.search(raw), END_MARK.search(raw)
            if s and e and e.start() > s.end():
                raw = raw[s.end():e.start()]
            path.write_text(raw.strip(), encoding="utf-8")
        docs.append({"title": title, "url": url,
                     "text": path.read_text(encoding="utf-8", errors="ignore")})

    # 2. Chunk each book into overlapping word windows (id like "frankenstein#7").
    chunks = []
    for d in docs:
        doc_id = re.sub(r"[^a-z0-9]+", "-", d["title"].lower()).strip("-")
        words = re.sub(r"\s+", " ", d["text"]).strip().split()
        step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)
        i = 0
        for start in range(0, len(words), step):
            window = words[start:start + WORDS_PER_CHUNK]
            if len(window) < max(60, WORDS_PER_CHUNK // 4):
                break
            chunks.append({"id": f"{doc_id}#{i}", "doc_id": doc_id, "chunk_index": i,
                           "title": d["title"], "source": d["url"], "text": " ".join(window)})
            i += 1
            if start + WORDS_PER_CHUNK >= len(words):
                break

    # 3. Embed every chunk and store it in Chroma (the slow, one-time step).
    batches = range(0, len(chunks), EMBED_BATCH_SIZE)
    if tqdm:
        batches = tqdm(batches, desc=f"Embedding with {EMBED_MODEL}")
    for sidx in batches:
        batch = chunks[sidx:sidx + EMBED_BATCH_SIZE]
        texts = [c["text"] for c in batch]
        resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                            json={"model": EMBED_MODEL, "input": texts}, timeout=600)
        resp.raise_for_status()
        vectors = resp.json()["embeddings"]
        collection.add(ids=[c["id"] for c in batch], documents=texts,
                       metadatas=[{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"],
                                   "title": c["title"], "source": c["source"]} for c in batch],
                       embeddings=vectors)

    already_built = True
    print(f"\nDone — Chroma now holds {collection.count()} vectors.")

Reusing 'rag_demo' — 8509 vectors already on disk. Nothing to build.


In [3]:
# --- the toolkit: the core notebook's steps, packaged as 5 small helpers ---
def ollama_chat(messages, temperature=0.2):
    """Call the local chat model and return its text reply."""
    r = SESSION.post(f"{OLLAMA_URL}/api/chat",
                     json={"model": CHAT_MODEL, "messages": messages,
                           "stream": False, "options": {"temperature": temperature}},
                     timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]


def dense_search(query, k=TOPK, where=None):
    """Embed `query`, return up to k Chroma hits (optionally filtered by `where`)."""
    r = SESSION.post(f"{OLLAMA_URL}/api/embed",
                     json={"model": EMBED_MODEL, "input": [query]}, timeout=600)
    r.raise_for_status()
    q_vec = r.json()["embeddings"][0]
    res = collection.query(query_embeddings=[q_vec], n_results=k, where=where,
                           include=["documents", "metadatas", "distances"])
    hits = []
    for id_, doc, meta, dist in zip(res["ids"][0], res["documents"][0],
                                    res["metadatas"][0], res["distances"][0]):
        meta = meta or {}
        hits.append({"id": id_, "text": doc or "", "title": meta.get("title", ""),
                     "doc_id": meta.get("doc_id", ""), "chunk_index": meta.get("chunk_index", -1),
                     "distance": float(dist)})
    return hits


def build_context(hits):
    """Stitch hits into one context block, each tagged with [doc_id#chunk_index]."""
    return "".join(f"[{h['doc_id']}#{h['chunk_index']}] {h['title']}\n{h['text']}\n---\n"
                   for h in hits) or "NO_CONTEXT"


SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say: 'I don't know based on the provided context.' "
    "Cite sources like [doc_id#chunk_index] for each key claim."
)


def generate(question, hits, temperature=0.2):
    """Answer `question` grounded only in `hits`."""
    user = f"Question: {question}\n\nContext:\n{build_context(hits)}"
    return ollama_chat([{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user}], temperature)


def show_hits(hits, key="distance"):
    """Pretty-print a hit list; `key` picks which score to display."""
    for i, h in enumerate(hits, 1):
        score = f"  {key}={h[key]:.3f}" if key in h and h[key] is not None else ""
        print(f"  {i}. {h['title']} [{h['doc_id']}#{h['chunk_index']}]{score}")

In [4]:
# Sanity check: the plain dense retriever every variant below builds on.
show_hits(dense_search("Who is Mr. Darcy?"))

  1. Pride and Prejudice [pride-and-prejudice#30]  distance=0.545
  2. Pride and Prejudice [pride-and-prejudice#47]  distance=0.554
  3. Pride and Prejudice [pride-and-prejudice#31]  distance=0.561
  4. Pride and Prejudice [pride-and-prejudice#495]  distance=0.587
  5. Pride and Prejudice [pride-and-prejudice#38]  distance=0.588


In [8]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

# 1. Meta Data Filtering

Select * from collection where title='Pride and Prejudice' order by distance limit 5;

In [9]:
query = "a mysterious creature comes to life"

print("Unfiltered (any book):")
show_hits(dense_search(query, k=3))

print("\nScoped to 'Frankenstein' only:")
show_hits(dense_search(query, k=3, where={"title": "Frankenstein"}))

Unfiltered (any book):
  1. Frankenstein [frankenstein#92]  distance=0.590
  2. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#420]  distance=0.600
  3. Dracula [dracula#91]  distance=0.613

Scoped to 'Frankenstein' only:
  1. Frankenstein [frankenstein#92]  distance=0.590
  2. Frankenstein [frankenstein#61]  distance=0.617
  3. Frankenstein [frankenstein#16]  distance=0.622


# 2. Query Transformation

## Multi Query

In [10]:
question = "Elizabeth's changing feelings about Darcy's proposal"

prompt = (f"Generate 3 alternative search queries (different wording / aspects) for the "
          f"question. One per line, no numbering.\n\nQuestion: {question}")


raw = ollama_chat([{"role": "user", "content": prompt}], temperature=0.5)
variants = ([question] + [ln.strip("-* ").strip() for ln in raw.splitlines() if ln.strip()])[:4]

print("Search queries:")
for v in variants:
    print("  -", v)

best = {}                                    # id -> best (lowest-distance) hit across all variants
for v in variants:
    for h in dense_search(v, k=5):
        if h["id"] not in best or h["distance"] < best[h["id"]]["distance"]:
            best[h["id"]] = h

print("\nUnion of results:")
show_hits(sorted(best.values(), key=lambda h: h["distance"])[:TOPK])

Search queries:
  - Elizabeth's changing feelings about Darcy's proposal
  - What motivates Elizabeth's shift in opinion towards Darcy's proposal in Pride and Prejudice?
  - Elizabeth's emotional response to Darcy's marriage proposal in Pride and Prejudice
  - How does Elizabeth's perception of Darcy change after he proposes to her in Pride and Prejudice

Union of results:
  1. Pride and Prejudice [pride-and-prejudice#271]  distance=0.396
  2. Pride and Prejudice [pride-and-prejudice#272]  distance=0.430
  3. Pride and Prejudice [pride-and-prejudice#267]  distance=0.465
  4. Pride and Prejudice [pride-and-prejudice#270]  distance=0.480
  5. Pride and Prejudice [pride-and-prejudice#457]  distance=0.481


## HyDE - Hypothetical Document Embeddings

You make the LLM imagine what the answer to the question might look like, and then embed that imagined answer to get a better query embedding for retrieval.

In [ ]:
question = "How does the creature feel toward Victor Frankenstein?"

hypo = ollama_chat([{"role": "user",
                     "content": f"Write a short, factual paragraph answering: {question}"}],
                   temperature=0.3)
print("Hypothetical answer we'll search with:")
pretty_print(hypo)

show_hits(dense_search(hypo))

Hypothetical answer we'll search with:
The creature in Mary Shelley's novel "Frankenstein" feels a complex mix of
emotions towards Victor Frankenstein, its creator. Initially, the creature is
grateful to Frankenstein for bringing it into existence, but as it becomes aware
of its own isolation and rejection by its creator, it develops a deep sense of
resentment and anger towards Frankenstein. The creature feels betrayed by
Frankenstein's abandonment and lack of acceptance, and it eventually seeks
revenge against its creator, leading to a tragic confrontation between the two.
Throughout the novel, the creature's feelings towards Frankenstein oscillate
between a desire for companionship and acceptance, and a fierce determination to
punish its creator for its suffering.


## Break down question into sub-questions

In [12]:
question = "Compare how Darcy and Heathcliff express love."

decomp = ollama_chat([{"role": "user", "content":
    f"Break this into up to 3 simpler standalone sub-questions, one per line. "
    f"If already simple, return it unchanged.\n\nQuestion: {question}"}], temperature=0.3)
print("Decomposed into sub-questions:")
for ln in decomp.splitlines():
    if ln.strip():
        print("  -", ln.strip("-*0123456789. ").strip())

stepback = ollama_chat([{"role": "user", "content":
    f"Give ONE broader background question behind this one:\n{question}"}], temperature=0.3)
print("\nStep-back question:", stepback.strip().splitlines()[0])

Decomposed into sub-questions:
  - Here are the sub-questions:
  - How does Darcy express love in the novel?
  - How does Heathcliff express love in the novel?
  - What are the similarities and differences in how Darcy and Heathcliff express love?

Step-back question: Here's a broader background question that could lead to this comparison:


# Re-Ranking

In [13]:
# Try a real cross-encoder; fall back to the LLM reranker if the dep is missing.
try:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    HAVE_CROSS_ENCODER = True
    print("Cross-encoder loaded.")
except Exception as e:
    cross_encoder, HAVE_CROSS_ENCODER = None, False
    print(f"Cross-encoder unavailable ({type(e).__name__}) — will use the LLM reranker.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder loaded.


In [14]:
query = "the detective examines a battered hat for clues about its owner"
pool = dense_search(query, k=12)          # retrieve wide, then re-rank down

print("Before re-ranking (dense top-5):")
show_hits(pool[:5])

if HAVE_CROSS_ENCODER:
    # Cross-encoder scores each (query, chunk) pair jointly.
    ce_scores = cross_encoder.predict([(query, h["text"]) for h in pool])
    order = np.argsort(ce_scores)[::-1][:TOPK]
    reranked = [{**pool[i], "rerank": float(ce_scores[i])} for i in order]
else:
    # LLM reranker: one tiny 0-10 rating call per candidate (keep the pool small).
    reranked = []
    for h in pool[:8]:
        out = ollama_chat([{"role": "user", "content":
            f"Rate 0-10 how well this passage helps answer the question. Reply with ONLY a number.\n"
            f"Question: {query}\nPassage: {h['text'][:1200]}"}], temperature=0.0)
        m = re.search(r"\d+(?:\.\d+)?", out)
        reranked.append({**h, "rerank": float(m.group()) if m else 0.0})
    reranked = sorted(reranked, key=lambda h: -h["rerank"])[:TOPK]

print("\nAfter re-ranking (top-5):")
show_hits(reranked, "rerank")

Before re-ranking (dense top-5):
  1. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#216]  distance=0.411
  2. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#219]  distance=0.504
  3. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#89]  distance=0.517
  4. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#217]  distance=0.517
  5. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#214]  distance=0.535

After re-ranking (top-5):
  1. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#214]  rerank=0.041
  2. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#215]  rerank=-0.065
  3. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#216]  rerank=-1.389
  4. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#217]  rerank=-7.592
  5. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#218]  rerank=-7.772


# Conversational RAG

In [15]:
history = []          # list of (role, text) turns
followups = ["Who is Mr. Darcy?",
             "Who does he eventually marry?",
             "Why did she dislike him at first?"]

for question in followups:
    # 1. Condense the follow-up into a standalone query (resolve pronouns) using the history.
    if history:
        convo = "\n".join(f"{role}: {txt}" for role, txt in history)
        prompt = ("Rewrite the follow-up as a standalone question (resolve pronouns/references). "
                  "Return ONLY the rewritten question.\n\n"
                  f"Conversation:\n{convo}\n\nFollow-up: {question}")
        standalone = ollama_chat([{"role": "user", "content": prompt}], temperature=0.0).strip()
    else:
        standalone = question

    # 2. Retrieve on the standalone query, 3. answer, 4. remember the turn.
    answer = generate(standalone, dense_search(standalone))
    history += [("user", question), ("assistant", answer)]

    print(f"\nYou: {question}")
    print(f"   (retrieved as -> {standalone})")
    print(textwrap.fill(f"Bot: {answer}", width=88))


You: Who is Mr. Darcy?
   (retrieved as -> Who is Mr. Darcy?)
Bot: Mr. Darcy is a character in the novel "Pride and Prejudice" who is described as
proud, haughty, and fastidious. He is a wealthy gentleman with a large estate in
Derbyshire and is considered to be one of the most eligible bachelors in the county. He
is initially portrayed as being dismissive and critical of those around him,
particularly the Bennet family, and is seen as being above his company. However, as the
novel progresses, his character is revealed to be more complex, and he is shown to be
capable of change and growth.  [pride-and-prejudice#30] describes him as "the proudest,
most disagreeable man in the world" and notes that he is "above his company, and above
being pleased". [pride-and-prejudice#31] shows him to be critical of Elizabeth Bennet,
saying that she is "tolerable, but not handsome enough to tempt me".  However, as the
novel progresses, it is revealed that Mr. Darcy has a more nuanced character and is


# Agentic RAG

In [17]:
# retrieve → GRADE → good?  ──yes──> answer
#                       │
#                       no
#                       ▼
#                reformulate → retrieve again → GRADE again → good? ──yes──> answer
#                                                              │
#                                                              no
#                                                              ▼
#                                                           ABSTAIN

In [18]:
# Pattern 7, now INSTRUMENTED: the decision logic is identical to before, but we
# print every intermediate step so you can WATCH grade -> reformulate -> abstain unfold.
GRADE_TMPL = ("Is this context relevant AND sufficient to answer the question? "
              "Reply ONLY YES or NO.\nQuestion: {q}\nContext:\n{ctx}")

for question in ["Who is Sherlock Holmes?", "What is the capital of France?"]:
    print("=" * 88)
    print(f"Q: {question}\n")

    # 1. First retrieval — what did the vector search actually pull back?
    hits = dense_search(question)
    print("[retrieve #1] top hits (lower distance = closer):")
    show_hits(hits)
    print(f"[context #1]  grader will read: {build_context(hits)[:140].strip()!r} ...\n")

    # 2. Grade #1 — is that context good enough? (show the model's RAW reply before parsing)
    grade = ollama_chat([{"role": "user",
        "content": GRADE_TMPL.format(q=question, ctx=build_context(hits)[:2000])}], temperature=0.0)
    good = grade.strip().upper().startswith("Y")
    print(f"[grade #1]    model said {grade.strip()!r}  ->  {'GOOD' if good else 'WEAK'}\n")

    if good:
        print("[decision]    context is good -> answer directly")
        route, answer = "direct", generate(question, hits)
    else:
        # 3. Corrective step: ask the LLM to rewrite the query, then retrieve again.
        print("[decision]    context is weak -> reformulate and retry")
        new_q = ollama_chat([{"role": "user",
            "content": f"Rewrite this to retrieve better passages: {question}"}], temperature=0.2).strip()
        print(f"[rewrite]     {question!r}\n              -> {new_q!r}")

        hits = dense_search(new_q)
        print("[retrieve #2] top hits on the reformulated query:")
        show_hits(hits)

        # 4. Grade #2 — did the retry actually help?
        grade2 = ollama_chat([{"role": "user",
            "content": GRADE_TMPL.format(q=question, ctx=build_context(hits)[:2000])}], temperature=0.0)
        good2 = grade2.strip().upper().startswith("Y")
        print(f"[grade #2]    model said {grade2.strip()!r}  ->  {'GOOD' if good2 else 'WEAK'}")

        if good2:
            print("[decision]    retry worked -> answer on the reformulated retrieval")
            route, answer = "reformulated", generate(question, hits)
        else:
            print("[decision]    still weak -> ABSTAIN (don't hallucinate)")
            route, answer = "abstain", "I don't know based on the available documents."

    print(f"\nROUTE: {route}")
    print(textwrap.fill(f"A: {answer}", width=88))
    print()

Q: Who is Sherlock Holmes?

[retrieve #1] top hits (lower distance = closer):
  1. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#236]  distance=0.539
  2. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#0]  distance=0.548
  3. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#337]  distance=0.548
  4. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#39]  distance=0.551
  5. Adventures of Sherlock Holmes [adventures-of-sherlock-holmes#180]  distance=0.551
[context #1]  grader will read: '[adventures-of-sherlock-holmes#236] Adventures of Sherlock Holmes\nto you.” “You? Who are you? How could you know anything of the matter?” “M' ...

[grade #1]    model said 'YES'  ->  GOOD

[decision]    context is good -> answer directly

ROUTE: direct
A: Sherlock Holmes is a detective who solves crimes and mysteries. He is a skilled
observer and uses his powers of deduction to gather information and piece together
clues. He is described as having a "c